# Laboratório 07: redes neurais "do zero"

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

Hoje não tem biblioteca de redes neurais: só **NumPy** (contas) e **Matplotlib**
(gráficos). Vamos construir, com as próprias mãos, as peças da aula: um neurônio, o
perceptron e a sua regra de aprendizado, o MLP que resolve o XOR e, por fim, um MLP
que **aprende sozinho** o XOR com retropropagação. Na próxima aula, uma biblioteca
(Keras) fará tudo isso em poucas linhas; hoje é para entender o que ela faz por dentro.

### Onde rodar

* **No laboratório (o normal hoje):** no projeto da disciplina, rode `uv run jupyter lab`
  e abra este arquivo. Ele só usa NumPy e Matplotlib, que já estão no projeto.
* **Em casa:** o mesmo, no seu clone do projeto (`uv sync` antes, se for a primeira vez).
* **No Google Colab (opcional):** em colab.research.google.com, *Arquivo → Fazer upload
  de notebook* e escolha este arquivo. Não precisa de nenhum arquivo extra. GPU é
  opcional e aqui não faz diferença (as contas são pequenas); se quiser ligar:
  *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU*.

| Parte | O que |
|---|---|
| 1 | um neurônio |
| 2 | a regra do perceptron aprendendo o E |
| 3 | o OU (aprende) e o XOR (não aprende) |
| 4 | as retas do perceptron |
| 5 | o MLP do livro resolve o XOR (pesos dados) |
| 6 | as funções de ativação |
| 7 | um MLP do zero, treinado com retropropagação |
| 8 | experimentos: sem não linearidade, pesos iguais, outras sementes |

As lacunas estão marcadas com `# TODO`. Cada uma tem uma dica e um valor provisório,
para o notebook rodar inteiro mesmo antes de você preencher. Depois de preencher um
TODO, rode de novo todas as células dali para baixo (*Run > Run Selected Cell and All
Below*). Os números
**(esperado: ...)** são os que você deve ver depois de preencher: a semente é fixa, e as
contas são as mesmas da aula. Na Parte 7, se a sua máquina tiver outra versão do NumPy,
a última casa decimal pode variar; o que não pode mudar é a rede acertar os quatro
casos do XOR.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEMENTE = 20261104   # a data da aula da turma B, como nos labs anteriores
np.set_printoptions(precision=4, suppress=True)


def lista(v, casas):
    """Mostra uma lista de numeros com o mesmo numero de casas decimais."""
    return "[" + ", ".join(f"{x:.{casas}f}" for x in v) + "]"

## Parte 1: um neurônio

Um neurônio faz duas coisas: a **soma ponderada** das entradas mais o viés,
$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$, e depois aplica a **ativação**. Aqui, a
função degrau: 1 se $z \ge 0$, senão 0 (Géron 2019, eq. 10-1, p. 264).

In [ ]:
def degrau(z):
    """Funcao degrau (Heaviside, Geron eq. 10-1): 0 se z < 0, 1 se z >= 0."""
    return np.where(np.asarray(z) >= 0, 1, 0)

print(degrau(-0.3), degrau(0), degrau(2.5))   # 0 1 1

In [ ]:
# TODO 1: complete a soma ponderada do neurônio.
# Dica: np.dot(w, x) multiplica entrada por entrada e soma tudo. Não esqueça o viés b.
def neuronio(x, w, b):
    """Soma ponderada mais vies, depois o degrau."""
    z = 0.0   # valor provisório: troque pela soma ponderada mais o viés
    return z, degrau(z)

In [ ]:
# o exemplo do slide "À mão: a saída de um neurônio"
x = np.array([2, 1, 3])
w = np.array([0.5, -1, 0.25])
b = -0.5
z, saida = neuronio(x, w, b)
print(f"z: {z:.2f}   (esperado: 0.25)")
print(f"saída: {int(saida)}   (esperado: 1)")

**Teste você:** troque `b = -0.5` por `b = -1` e rode de novo. O que acontece com a
saída? (No slide: $z = -0{,}25$, saída 0.)

## Parte 2: a regra do perceptron aprendendo o E

Os quatro exemplos das portas lógicas, sempre nesta ordem: $(0,0)$, $(0,1)$, $(1,0)$,
$(1,1)$.

In [ ]:
X_logica = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_e   = np.array([0, 0, 0, 1])     # E (AND)
y_ou  = np.array([0, 1, 1, 1])     # OU (OR)
y_xou = np.array([0, 1, 1, 0])     # OU exclusivo (XOR)
Y_xou = y_xou.reshape(-1, 1).astype(float)   # o mesmo XOR, em coluna, para o MLP

A **regra do perceptron** (Géron 2019, eq. 10-2, p. 265), para cada exemplo, um de
cada vez:

$$w_i \leftarrow w_i + \eta\,(y - \hat{y})\,x_i \qquad\qquad b \leftarrow b + \eta\,(y - \hat{y})$$

Se acertou, $y - \hat{y} = 0$ e nada muda. Uma **época** é uma passada pelos quatro
exemplos. O treino para na primeira época sem nenhum erro.

In [ ]:
def treinar_perceptron(X, y, w, b, taxa, epocas):
    """Regra do perceptron (Geron eq. 10-2), um exemplo de cada vez.
    Devolve: pesos finais, vies final, erros em cada epoca, as atualizacoes
    feitas (w e b depois de cada uma) e a epoca em que nao errou nada."""
    w = np.array(w, dtype=float)
    erros_por_epoca, atualizacoes, parou = [], [], None
    for epoca in range(1, epocas + 1):
        erros = 0
        for xi, yi in zip(X, y):
            y_hat = degrau(np.dot(w, xi) + b)
            erro = yi - y_hat
            if erro != 0:
                erros += 1
                # TODO 2: aplique a regra do perceptron em w e em b.
                # Dica: w = w + taxa * erro * xi   e   b = b + taxa * erro
                # (valor provisório: sem as duas linhas, o perceptron nunca muda)
                atualizacoes.append((w.copy(), b))
        erros_por_epoca.append(erros)
        if erros == 0:
            parou = epoca
            break
    return w, b, erros_por_epoca, atualizacoes, parou

In [ ]:
# o mesmo começo do slide "À mão: uma atualização"
w, b, erros, atualizacoes, parou = treinar_perceptron(X_logica, y_e, w=[0.25, 0.5], b=-1.0,
                                                      taxa=0.5, epocas=20)
w1a, w2a, ba = (atualizacoes[0][0][0], atualizacoes[0][0][1], atualizacoes[0][1]) if atualizacoes else (np.nan,) * 3
print(f"1ª atualização, w1: {w1a:.2f}   (esperado: 0.75)")
print(f"1ª atualização, w2: {w2a:.2f}   (esperado: 1.00)")
print(f"1ª atualização, b: {ba:.2f}   (esperado: -0.50)")
print("erros por época:", erros)
print(f"época em que acertou tudo: {parou}   (esperado: 3)")
print(f"w final: {lista(w, 2)}   (esperado: [0.75, 0.50])")
print(f"b final: {b:.2f}   (esperado: -1.00)")

Confira no papel: com $w = (0{,}75;\ 0{,}5)$ e $b = -1$, só o $(1,1)$ dá
$z \ge 0$.

## Parte 3: o OU aprende; o XOR, não

In [ ]:
w_ou, b_ou, erros_ou, _, parou_ou = treinar_perceptron(X_logica, y_ou, w=[0.25, 0.5], b=-1.0,
                                                       taxa=0.5, epocas=20)
print("OU, erros por época:", erros_ou)
print(f"OU, época em que acertou tudo: {parou_ou}   (esperado: 3)")
print(f"OU, w final: {lista(w_ou, 2)}   (esperado: [0.75, 1.00])")
print(f"OU, b final: {b_ou:.2f}   (esperado: -0.50)")

In [ ]:
w_x, b_x, erros_x, _, parou_x = treinar_perceptron(X_logica, y_xou, w=[0.25, 0.5], b=-1.0,
                                                   taxa=0.5, epocas=20)
print("XOR, erros por época:", erros_x)
print(f"XOR, época em que acertou tudo: {parou_x}   (esperado: None)")
print(f"XOR, menor número de erros numa época: {min(erros_x)}   (esperado: 2)")

plt.figure(figsize=(6, 3))
plt.plot(range(1, len(erros_x) + 1), erros_x, "o-")
plt.xlabel("época"); plt.ylabel("erros na época"); plt.ylim(-0.2, 4.2)
plt.title("perceptron no XOR: nunca chega a zero"); plt.grid(alpha=.3); plt.show()

In [ ]:
# TODO 3: quantos dos 4 exemplos do XOR o perceptron final (w_x, b_x) acerta?
# Dica: previsoes = degrau(X_logica @ w_x + b_x); compare com y_xou e conte os iguais com .sum()
acertos = 0   # valor provisório

print(f"acertos no XOR depois de 20 épocas: {acertos}   (esperado: 2)")

O perceptron fica girando em círculo: corrige um exemplo e estraga outro. Não é falta
de épocas: **nenhuma reta** separa o XOR (a demonstração com as quatro desigualdades
está no slide "XOR: nenhuma reta separa").

## Parte 4: as retas do perceptron

A fronteira do perceptron é onde $w_1x_1 + w_2x_2 + b = 0$, isto é,
$x_2 = -(w_1x_1 + b)/w_2$.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
xs = np.linspace(-0.5, 1.5, 50)
for a, alvo, titulo, wb in [(ax[0], y_e, "E", (w, b)), (ax[1], y_ou, "OU", (w_ou, b_ou)),
                            (ax[2], y_xou, "XOR (perceptron final)", (w_x, b_x))]:
    a.scatter(X_logica[:, 0], X_logica[:, 1], c=alvo, cmap="RdYlGn", s=150, edgecolor="k", zorder=3)
    ww, bb = wb
    if ww[1] != 0:
        a.plot(xs, -(ww[0] * xs + bb) / ww[1], "k-")
    elif ww[0] != 0:
        a.axvline(-bb / ww[0], color="k")
    a.set_xlim(-0.5, 1.5); a.set_ylim(-0.5, 1.5); a.set_aspect("equal"); a.set_title(titulo)
plt.show()

## Parte 5: o MLP do livro resolve o XOR

A rede da figura 10-6 do Géron (p. 267): duas entradas, dois neurônios ocultos, uma
saída, todos com degrau. Os pesos já estão escolhidos (a figura marca só alguns;
lemos as setas sem número como peso 1, e a tabela da p. 266 confirma).

Para calcular os quatro exemplos de uma vez, usamos o produto de matrizes `@`:
`X_logica @ W1` faz, **para cada exemplo (linha)**, a soma ponderada de **cada neurônio
oculto (coluna)**. A conta do exemplo $(1,1)$ no primeiro neurônio, à mão:
$1 \cdot 1 + 1 \cdot 1 - 1{,}5 = 0{,}5$.

In [ ]:
W1 = np.array([[1, 1],
               [1, 1]])          # linha i: pesos que saem da entrada x_i; coluna j: neurônio oculto j
b1 = np.array([-1.5, -0.5])      # vieses dos dois neurônios ocultos
W2 = np.array([[-1],
               [1]])             # pesos dos dois ocultos para a saída
b2 = np.array([-0.5])

print("somas da camada oculta (cada linha é um exemplo):")
print(X_logica @ W1 + b1)

In [ ]:
# TODO 4: o forward pass do MLP do livro.
# Dica: H = degrau(X_logica @ W1 + b1)   e depois   saida = degrau(H @ W2 + b2)
H = np.zeros((4, 2), dtype=int)       # valor provisório
saida = np.zeros((4, 1), dtype=int)   # valor provisório

print("camada oculta (h1 = E, h2 = OU):")
print(H)
print(f"saídas: {saida.ravel().tolist()}   (esperado: [0, 1, 1, 0])")

A primeira coluna de `H` é o **E**; a segunda, o **OU**. A saída é "OU, mas não E":
o XOR. A camada oculta transformou os pontos, e no espaço novo uma reta separa.

## Parte 6: as funções de ativação

As quatro da figura 10-8 do Géron (p. 269) e do notebook oficial do cap. 10: degrau,
sigmoide $\sigma(z) = 1/(1 + e^{-z})$, tanh e ReLU $= \max(0, z)$.

In [ ]:
# TODO 5: complete a sigmoide e a ReLU.
# Dica: sigmoide: 1 / (1 + np.exp(-z))      ReLU: np.maximum(0, z)
def sigmoide(z):
    return z * 0 + 0.5   # valor provisório: devolve sempre 0,5

def relu(z):
    return z * 0         # valor provisório: devolve sempre 0

In [ ]:
print(f"sigmoide(-2): {sigmoide(-2):.4f}   (esperado: 0.1192)")
print(f"sigmoide(0): {sigmoide(0):.4f}   (esperado: 0.5000)")
print(f"sigmoide(2): {sigmoide(2):.4f}   (esperado: 0.8808)")
print(f"tanh(-2): {np.tanh(-2):.4f}   (esperado: -0.9640)")
print(f"tanh(2): {np.tanh(2):.4f}   (esperado: 0.9640)")
print(f"relu(-2): {relu(-2):.4f}   (esperado: 0.0000)")
print(f"relu(2): {relu(2):.4f}   (esperado: 2.0000)")

z = np.linspace(-4, 4, 401)
fig, ax = plt.subplots(1, 4, figsize=(12, 2.8))
for a, (nome, f) in zip(ax, [("degrau", degrau), ("sigmoide", sigmoide), ("tanh", np.tanh), ("ReLU", relu)]):
    a.plot(z, f(z)); a.axhline(0, color="gray", lw=.5); a.axvline(0, color="gray", lw=.5)
    a.set_title(nome); a.grid(alpha=.3)
plt.show()

A sigmoide e a tanh têm inclinação em todo lugar; a degrau é plana (inclinação zero),
e por isso não serve para treinar com gradiente (Géron 2019, p. 268).

## Parte 7: um MLP do zero, treinado com retropropagação

Agora ninguém escolhe os pesos: a rede aprende o XOR sozinha.

* **Rede:** 2 entradas → 2 neurônios ocultos (sigmoide) → 1 saída (sigmoide). São 9
  números para ajustar: 4 pesos e 2 vieses na oculta, 2 pesos e 1 viés na saída.
* **Perda:** o MSE da aula de regressão.
* **Treino:** gradiente descendente, com a retropropagação calculando a culpa de cada
  peso, da saída para a entrada.

O laço de cada época tem quatro passos: **ida**, **perda**, **volta**, **passo**.

In [ ]:
def iniciar(semente):
    """Pesos sorteados (numeros pequenos, em torno de 0); vieses comecam em 0."""
    rng = np.random.default_rng(semente)
    return {"W1": rng.normal(0, 1, size=(2, 2)), "b1": np.zeros(2),
            "W2": rng.normal(0, 1, size=(2, 1)), "b2": np.zeros(1)}

def perda_mse(Y_hat, Y):
    return np.mean((Y_hat - Y) ** 2)

p0 = iniciar(SEMENTE)
print("pesos sorteados da camada oculta:\n", p0["W1"])

In [ ]:
# TODO 6: o forward pass (a ida) do MLP.
# Dica: H = sigmoide(X @ p["W1"] + p["b1"])   e   Y_hat = sigmoide(H @ p["W2"] + p["b2"])
def ida(p, X):
    """Forward pass: entrada -> oculta (sigmoide) -> saida (sigmoide)."""
    H = np.full((len(X), 2), 0.5)       # valor provisório
    Y_hat = np.full((len(X), 1), 0.5)   # valor provisório
    return H, Y_hat

A **volta** já vem pronta, com um comentário por linha. Leia junto com o slide "À
mão: a culpa de cada peso": a culpa de um peso é a culpa que chega ao neurônio vezes a
entrada daquele peso; para passar para a camada de trás, a culpa é multiplicada pelo
peso que liga as camadas e pela inclinação da ativação. A inclinação da sigmoide, num
ponto em que ela vale $s$, é $s\,(1 - s)$ (é um fato da função, conferido numericamente
pelo gerador da aula; não precisa decorar).

In [ ]:
def volta(p, X, Y, H, Y_hat):
    """Retropropagacao: a culpa (o gradiente) de cada peso, da saida para a entrada."""
    # 1. culpa que chega a saida: a do MSE vezes a inclinacao da sigmoide da saida
    culpa_saida = 2 * (Y_hat - Y) / len(X) * Y_hat * (1 - Y_hat)
    # 2. culpa dos pesos da saida: culpa que chega x a entrada desses pesos (H)
    g = {"W2": H.T @ culpa_saida, "b2": culpa_saida.sum(axis=0)}
    # 3. a culpa volta pelos pesos W2 ate a camada oculta, vezes a inclinacao da sigmoide
    culpa_oculta = (culpa_saida @ p["W2"].T) * H * (1 - H)
    # 4. culpa dos pesos da oculta: culpa que chega x a entrada desses pesos (X)
    g["W1"] = X.T @ culpa_oculta
    g["b1"] = culpa_oculta.sum(axis=0)
    return g

In [ ]:
def treinar(semente, taxa, epocas, iniciar=None, ida=None, volta=None):
    # sem argumento, usa as versoes ATUAIS de iniciar, ida e volta (as do seu TODO)
    iniciar = iniciar or globals()["iniciar"]
    ida = ida or globals()["ida"]
    volta = volta or globals()["volta"]
    p = iniciar(semente)
    perdas = []
    for epoca in range(1, epocas + 1):
        H, Y_hat = ida(p, X_logica)                  # ida
        perdas.append(perda_mse(Y_hat, Y_xou))       # perda
        g = volta(p, X_logica, Y_xou, H, Y_hat)      # volta (retropropagacao)
        for nome in p:                               # passo do gradiente descendente
            # TODO 7: o passo do gradiente descendente, igual ao da regressão.
            # Dica: p[nome] = p[nome] - taxa * g[nome]
            pass   # valor provisório: sem o passo, a rede nunca aprende
    return p, perdas

In [ ]:
TAXA = 1.0
EPOCAS = 5000
p, perdas = treinar(SEMENTE, TAXA, EPOCAS)
H, Y_hat = ida(p, X_logica)
perda_final = perda_mse(Y_hat, Y_xou)
acertou = bool(np.array_equal((Y_hat.ravel() > 0.5).astype(int), y_xou))

print(f"perda na 1ª época: {perdas[0]:.4f}   (esperado: 0.3161)")
print(f"perda final: {perda_final:.4f}   (esperado: 0.0006)")
print(f"respostas da rede: {lista(Y_hat.ravel(), 2)}   (esperado: [0.03, 0.97, 0.98, 0.02])")
print(f"acertou os 4 casos do XOR: {acertou}   (esperado: True)")
print(f"perda final abaixo de 0,01: {bool(perda_final < 0.01)}   (esperado: True)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(range(1, len(perdas) + 1), perdas)
ax[0].set_xlabel("época"); ax[0].set_ylabel("perda (MSE)"); ax[0].set_title("a perda a cada época")
ax[0].grid(alpha=.3)
g1, g2 = np.meshgrid(np.linspace(-0.25, 1.25, 151), np.linspace(-0.25, 1.25, 151))
_, saida_grade = ida(p, np.c_[g1.ravel(), g2.ravel()])
cs = ax[1].contourf(g1, g2, saida_grade.reshape(g1.shape), levels=np.linspace(0, 1, 11), cmap="RdYlGn")
um, zero = y_xou == 1, y_xou == 0
ax[1].scatter(X_logica[um, 0], X_logica[um, 1], marker="o", c="white", s=150, edgecolor="k", label="XOR = 1")
ax[1].scatter(X_logica[zero, 0], X_logica[zero, 1], marker="s", c="black", s=150, label="XOR = 0")
ax[1].legend(loc="upper center", fontsize=8, ncol=2)
ax[1].set_aspect("equal"); ax[1].set_title("saída da rede treinada (verde: perto de 1)"); fig.colorbar(cs, ax=ax[1])
plt.show()

**Leia o gráfico da perda:** da época 20 até perto da 200, a rede responde perto de
0,5 para todo mundo (perda perto de 0,25); depois ela "descobre" o caminho e a perda
despenca. À direita, a região aprendida: uma faixa diagonal, com **duas** fronteiras.
Uma reta só não daria conta.

## Parte 8: experimentos

**(a) Sem não linearidade.** A mesma rede, com a camada oculta **sem** a sigmoide (só a
soma ponderada). A aula disse que camadas lineares empilhadas viram uma camada só. Veja
acontecer.

In [ ]:
def ida_sem_ativacao(p, X):
    H = X @ p["W1"] + p["b1"]                 # camada oculta SEM ativacao
    Y_hat = sigmoide(H @ p["W2"] + p["b2"])
    return H, Y_hat

def volta_sem_ativacao(p, X, Y, H, Y_hat):
    culpa_saida = 2 * (Y_hat - Y) / len(X) * Y_hat * (1 - Y_hat)
    g = {"W2": H.T @ culpa_saida, "b2": culpa_saida.sum(axis=0)}
    culpa_oculta = culpa_saida @ p["W2"].T   # sem ativacao: a inclinacao e 1
    g["W1"] = X.T @ culpa_oculta
    g["b1"] = culpa_oculta.sum(axis=0)
    return g

p_lin, _ = treinar(SEMENTE, TAXA, EPOCAS, ida=ida_sem_ativacao, volta=volta_sem_ativacao)
_, Y_lin = ida_sem_ativacao(p_lin, X_logica)
print(f"sem ativação, perda final: {perda_mse(Y_lin, Y_xou):.4f}   (esperado: 0.2500)")
print(f"sem ativação, respostas: {lista(Y_lin.ravel(), 2)}   (esperado: [0.00, 0.99, 0.00, 0.00])")
print(f"sem ativação, acertou os 4: {bool(np.array_equal((Y_lin.ravel() > 0.5).astype(int), y_xou))}   (esperado: False)")

**(b) Pesos iniciais iguais.** Todos os pesos começam em 0,5 (Géron 2019, p. 272-273:
é preciso sortear "para evitar quaisquer simetrias").

In [ ]:
def iniciar_iguais(semente):
    """Todos os pesos comecam iguais (0,5): a semente nem e usada."""
    return {"W1": np.full((2, 2), 0.5), "b1": np.zeros(2),
            "W2": np.full((2, 1), 0.5), "b2": np.zeros(1)}

p_igu, _ = treinar(SEMENTE, TAXA, EPOCAS, iniciar=iniciar_iguais)
_, Y_igu = ida(p_igu, X_logica)
print("pesos da camada oculta depois do treino:\n", p_igu["W1"])
print(f"os dois ocultos continuam gêmeos: {bool(np.allclose(p_igu['W1'][:, 0], p_igu['W1'][:, 1]))}   (esperado: True)")
print(f"pesos iguais, respostas: {lista(Y_igu.ravel(), 3)}   (esperado: [0.040, 0.665, 0.665, 0.668])")
print(f"pesos iguais, acertou os 4: {bool(np.array_equal((Y_igu.ravel() > 0.5).astype(int), y_xou))}   (esperado: False)")

**(c) Outras sementes.** O sorteio inicial muda o resultado. Com as sementes 0 a 19,
quantas redes aprendem o XOR? (Demora alguns segundos.)

In [ ]:
aprenderam = 0
for s in range(20):
    ps, _ = treinar(s, TAXA, EPOCAS)
    _, Ys = ida(ps, X_logica)
    if np.array_equal((Ys.ravel() > 0.5).astype(int), y_xou) and perda_mse(Ys, Y_xou) < 0.01:
        aprenderam += 1
print(f"sementes (de 20) em que a rede aprendeu o XOR: {aprenderam}   (esperado: 16)")

As redes que não aprenderam ficaram presas antes de achar o caminho (como o k-means,
que para num mínimo local conforme o sorteio inicial).

## Para entregar a si mesmo (não vale nota)

1. Na Parte 2, por que o $w_1$ não mudou na segunda atualização?
2. Por que o perceptron da Parte 3 nunca para no XOR, mesmo com mais épocas?
3. Na Parte 5, o que calcula cada neurônio oculto, em palavras?
4. Na Parte 7, em que sentido anda a "culpa" na função `volta`, e por quê?
5. O que a Parte 8(a) mostra sobre a ativação?
6. Por que, na Parte 8(b), os dois neurônios ocultos terminam iguais?